## Cross-session Single-cell tuning summary analyses

This notebook loads results from single cell-level analyses (generated by single_cell_analysis.ipynb) from multiple imaging sessions across animals and performs statistical comparisons and visualizations.

Last edited: WMS 4/18/26

In [ ]:
# Imports
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

from session_config import *
from xsession_utils import *

#### define data path and determine whether to save out files

In [ ]:
sc_xsession_dir = os.path.join(CROSS_SESSION_RESULTS_DIR, "single_cell_summary")
os.makedirs(sc_xsession_dir, exist_ok=True)

In [ ]:
save_csvs=False

In [ ]:
save_figs=False

#### Load data and construct summary df

In [ ]:
all_data = load_all_single_cell_data(ANIMAL_PATHS)

In [ ]:
all_data['6175'][5]

In [ ]:
all_props = extract_tuning_proportions(all_data,categories=None,filter_mixed_by_within_trial=True,collapse_mixed_categories=True)
all_props

In [ ]:
#save out a copy
if save_csvs:
    all_props.to_csv(os.path.join(sc_xsession_dir,'tuning_proportions_all_animals_all_sessions.csv'),index=False)

Create a second version without the sessions with insufficient training data for decoding in case it's needed

In [ ]:
bad_decoding_sessions = pd.read_csv(r"C:\miniscope_files\analyzed\xsessionxanimal_results\population_metrics\decoding\pseudotrials_vs_decoding_performance\decoding_runs_below_training_data_threshold.csv",
        header=0)
bad_decoding_sessions

In [ ]:
bad_decoding_sessions_social=bad_decoding_sessions[bad_decoding_sessions['decode_type']=='identity_across'].copy()
bad_decoding_sessions_empty=bad_decoding_sessions[bad_decoding_sessions['decode_type']=='location_empty1_across'].copy()
bad_decoding_sessions_social

In [ ]:
social_categories = ['partner_cells', 'novel_cells', 'left_cells', 'right_cells',
                     'partner_left_mixed', 'partner_right_mixed', 
                     'novel_left_mixed', 'novel_right_mixed',
                     'partner_mixed', 'novel_mixed', 'left_mixed', 'right_mixed']

all_props['_key'] = list(zip(all_props['animal'].astype(str), all_props['session_idx']))

is_social_cat = all_props['category'].isin(social_categories)
is_empty_cat = all_props['category'].isin(['left_empty1_cells', 'right_empty1_cells'])

bad_social_keys = set(
    zip(bad_decoding_sessions_social['animal'].astype(str),
        bad_decoding_sessions_social['session_idx'])
)

# Same for empty
bad_empty_keys = set(
    zip(bad_decoding_sessions_empty['animal'].astype(str),
        bad_decoding_sessions_empty['session_idx'])
)

bad_social_mask = is_social_cat & all_props['_key'].isin(bad_social_keys)
bad_empty_mask = is_empty_cat & all_props['_key'].isin(bad_empty_keys)

all_props_filtered_social = all_props[~(bad_social_mask)].drop(columns='_key').reset_index(drop=True)
all_props_filtered_empty = all_props[~(bad_empty_mask)].drop(columns='_key').reset_index(drop=True)
all_props = all_props.drop(columns='_key') #clean up the key column

In [ ]:
all_props_filtered_social

In [ ]:
all_props_filtered_empty

In [ ]:
#save out a copy
if save_csvs:
    all_props_filtered_social.to_csv(os.path.join(sc_xsession_dir,'tuning_proportions_all_animals_good_social_trials_decoding_sessions.csv'),index=False)
    all_props_filtered_empty.to_csv(os.path.join(sc_xsession_dir,'tuning_proportions_all_animals_good_empty_trial_1_decoding_sessions.csv'),index=False)

### Proportions across sessions across animals

In [ ]:
tuning_xsession_dir = os.path.join(sc_xsession_dir, "tuning_type_props")
os.makedirs(tuning_xsession_dir, exist_ok=True)

Generate some summary dataframes - mean proportions of each category across animals at each time point

In [ ]:
mean_props_dir = os.path.join(tuning_xsession_dir, "mean_props")
os.makedirs(mean_props_dir, exist_ok=True)

In [ ]:
all_props

In [ ]:
summ_tuning_props = summarize_tuning_proportions(all_props)
summ_tuning_props

In [ ]:
summ_tuning_props_filtered_social = summarize_tuning_proportions(all_props_filtered_social)
summ_tuning_props_filtered_social

In [ ]:
if save_csvs:
    summ_tuning_props.to_csv(os.path.join(mean_props_dir,'mean_proportions_per_session_across_animals_all_sessions.csv'),index=False)
    summ_tuning_props_filtered_social.to_csv(os.path.join(mean_props_dir,'mean_proportions_per_session_across_animals_good_decoding_sessions.csv'),index=False)

Plot the total (or mean) proportion of each proportion across animals for each timepoint (with options to collapse categories into supertypes and/or show untuned cells)

In [ ]:
def plot_tuning_proportions_stacked(summary_df, mode='social',
                                     mixed_color_by='identity',
                                     collapse_to_supertypes=False,
                                     proportion_method='mean',
                                     show_errorbars=True,
                                     session_indices=None, figsize=(10, 6),
                                     show_n=True, show_untuned=False,
                                     save_path=None, show=True):
    """
    Stacked bar plot of tuning category proportions per session.
    
    Parameters
    ----------
    summary_df : pd.DataFrame
        Output from summarize_tuning_proportions(). Must contain columns:
        session_idx, short_label, category, mean_prop, sem_prop, n_animals.
        If proportion_method='total', must also contain total_n_tuned_cells,
        total_all_recorded_cells, and total_prop.
    mode : str
        'social' to plot social-trial categories, 'empty' for empty1.
    mixed_color_by : str
        'identity' or 'location'. Only applies when mode='social' and 
        collapse_to_supertypes=False.
    collapse_to_supertypes : bool
        If True (and mode='social'), collapses into three supertypes.
    proportion_method : str
        'mean' uses per-animal mean proportions with SEM error bars.
        'total' uses pooled proportion (total tuned / total recorded across 
        all animals). No error bars are shown in this mode since it is a 
        single pooled statistic rather than a mean ± SEM.
    show_errorbars : bool
        If True and proportion_method='mean', draw SEM error bars. 
        Ignored when proportion_method='total'.
    session_indices : list[int] or None
    figsize : tuple
    show_n : bool
        When proportion_method='mean', annotate with n_animals.
        When proportion_method='total', annotate with total recorded cells.
    show_untuned : bool
    save_path : str or None
    show : bool
    
    Returns
    -------
    fig, ax
    """
    
    if proportion_method not in ('mean', 'total'):
        raise ValueError(f"proportion_method must be 'mean' or 'total', got '{proportion_method}'")
    
    # Select which columns to use based on method
    prop_col = 'mean_prop' if proportion_method == 'mean' else 'total_prop'
    
    # Color palette
    UNTUNED_COLOR = '#D3D3D3'
    
    # Supertype grays
    IDENTITY_SUPERTYPE_COLOR = '#2E2E2E'
    LOCATION_SUPERTYPE_COLOR = '#6B6B6B'
    MIXED_SUPERTYPE_COLOR    = '#A8A8A8'
    
    # ---- Build category lists, labels, colors per mode ----
    if mode == 'social':
        if collapse_to_supertypes:
            supertype_groups = {
                'identity_supertype': ['partner_cells', 'novel_cells'],
                'location_supertype': ['left_cells', 'right_cells'],
                'mixed_supertype':    ['partner_left_mixed', 'partner_right_mixed',
                                       'novel_left_mixed', 'novel_right_mixed'],
            }
            categories = list(supertype_groups.keys())
            category_labels = ['Identity-tuned', 'Location-tuned', 'Mixed-selective']
            category_colors = [IDENTITY_SUPERTYPE_COLOR, LOCATION_SUPERTYPE_COLOR, 
                               MIXED_SUPERTYPE_COLOR]
            
            collapsed_rows = []
            for s_idx in summary_df['session_idx'].unique():
                sess_rows = summary_df[summary_df['session_idx'] == s_idx]
                if len(sess_rows) == 0:
                    continue
                short_lbl = sess_rows['short_label'].iloc[0]
                for super_name, source_cats in supertype_groups.items():
                    matching = sess_rows[sess_rows['category'].isin(source_cats)]
                    if len(matching) == 0:
                        continue
                    
                    row = {
                        'session_idx': s_idx,
                        'short_label': short_lbl,
                        'category': super_name,
                        'n_animals': matching['n_animals'].max(),
                    }
                    
                    if proportion_method == 'mean': #this method will be less accurate when combining across supergrops because you are summing across the mean proportions of different categories across animals 
                        row['mean_prop'] = matching['mean_prop'].sum()
                        row['sem_prop'] = np.sqrt((matching['sem_prop'] ** 2).sum())
                    else:
                        row['total_n_tuned_cells'] = matching['total_n_tuned_cells'].sum()
                        row['total_all_recorded_cells'] = matching['total_all_recorded_cells'].max()
                        row['total_prop'] = (row['total_n_tuned_cells'] / 
                                             row['total_all_recorded_cells'] 
                                             if row['total_all_recorded_cells'] > 0 else 0)
                    
                    collapsed_rows.append(row)
            df = pd.DataFrame(collapsed_rows)
        else:
            if mixed_color_by not in ('identity', 'location'):
                raise ValueError(f"mixed_color_by must be 'identity' or 'location', "
                                 f"got '{mixed_color_by}'")
            
            pure_categories = ['partner_cells', 'novel_cells', 'left_cells', 'right_cells']
            pure_labels = ['Partner', 'Novel', 'Left', 'Right']
            pure_colors = [PARTNER_COLOR, NOVEL_COLOR, LEFT_COLOR, RIGHT_COLOR]
            
            if mixed_color_by == 'identity':
                mixed_categories = ['partner_left_mixed', 'partner_right_mixed',
                                    'novel_left_mixed', 'novel_right_mixed']
                mixed_labels = ['Partner/Left', 'Partner/Right', 
                                'Novel/Left', 'Novel/Right']
                mixed_colors = [PARTNER_MIXED_COLOR, PARTNER_MIXED_COLOR, NOVEL_MIXED_COLOR, NOVEL_MIXED_COLOR]
            else:
                mixed_categories = ['partner_left_mixed', 'novel_left_mixed',
                                    'partner_right_mixed', 'novel_right_mixed']
                mixed_labels = ['Partner/Left', 'Novel/Left',
                                'Partner/Right', 'Novel/Right']
                mixed_colors = [LEFT_MIXED_COLOR, LEFT_MIXED_COLOR, RIGHT_MIXED_COLOR, RIGHT_MIXED_COLOR]
            
            categories = pure_categories + mixed_categories
            category_labels = pure_labels + mixed_labels
            category_colors = pure_colors + mixed_colors
            
            df = summary_df[summary_df['category'].isin(categories)].copy()
    
    elif mode == 'empty':
        categories = ['left_empty1_cells', 'right_empty1_cells']
        category_labels = ['Left (empty1)', 'Right (empty1)']
        category_colors = [LEFT_COLOR, RIGHT_COLOR]
        df = summary_df[summary_df['category'].isin(categories)].copy()
    
    else:
        raise ValueError(f"mode must be 'social' or 'empty', got '{mode}'")
    
    # ---- Filter sessions ----
    if session_indices is None:
        session_indices = sorted(df['session_idx'].unique())
    df = df[df['session_idx'].isin(session_indices)]
    
    if len(df) == 0:
        print("No data to plot")
        return None, None
    
    # Pivot using the selected proportion column
    means_pivot = df.pivot(index='session_idx', columns='category', values=prop_col).fillna(0)
    n_pivot = df.pivot(index='session_idx', columns='category', values='n_animals').fillna(0)
    
    if proportion_method == 'mean':
        sems_pivot = df.pivot(index='session_idx', columns='category', values='sem_prop').fillna(0)
    
    # For total method, pivot the total recorded cells for the n annotation
    if proportion_method == 'total':
        total_cells_pivot = df.pivot(index='session_idx', columns='category', 
                                      values='total_all_recorded_cells').fillna(0)
    
    available_cats = [c for c in categories if c in means_pivot.columns]
    means_pivot = means_pivot[available_cats].reindex(session_indices)
    if proportion_method == 'mean':
        sems_pivot = sems_pivot[available_cats].reindex(session_indices)
    
    plot_labels = [category_labels[categories.index(c)] for c in available_cats]
    plot_colors = [category_colors[categories.index(c)] for c in available_cats]
    
    x_labels = [SESSION_CONFIG['short_labels'][s] for s in session_indices]
    x_pos = np.arange(len(session_indices))
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Build stacks
    bottom = np.zeros(len(session_indices))
    for cat, label, color in zip(available_cats, plot_labels, plot_colors):
        values = means_pivot[cat].values
        ax.bar(x_pos, values, bottom=bottom, color=color, label=label,
               edgecolor='white', linewidth=0.5, width=0.7)
        if show_errorbars and proportion_method == 'mean':
            sems = sems_pivot[cat].values
            ax.errorbar(x_pos, bottom + values, yerr=sems, fmt='none',
                        ecolor='black', elinewidth=0.8, capsize=3, alpha=0.7)
        bottom += values
    
    # Add untuned segment last
    if show_untuned:
        untuned_values = np.clip(1.0 - bottom, 0, None)
        ax.bar(x_pos, untuned_values, bottom=bottom, color=UNTUNED_COLOR,
               label='Untuned', edgecolor='white', linewidth=0.5, width=0.7)
        bottom += untuned_values
    
    if show_n:
        for i, s_idx in enumerate(session_indices):
            if proportion_method == 'mean':
                n_vals = n_pivot.loc[s_idx, available_cats].values if s_idx in n_pivot.index else []
                n_max = int(max(n_vals)) if len(n_vals) > 0 else 0
                n_label = f'n={n_max} animals'
            else:
                total_vals = total_cells_pivot.loc[s_idx, available_cats].values if s_idx in total_cells_pivot.index else []
                n_total = int(max(total_vals)) if len(total_vals) > 0 else 0
                n_animals_vals = n_pivot.loc[s_idx, available_cats].values if s_idx in n_pivot.index else []
                n_animals = int(max(n_animals_vals)) if len(n_animals_vals) > 0 else 0
                n_label = f'{n_total} cells\n({n_animals} animals)'
            ax.text(x_pos[i], bottom[i] + 0.01, n_label,
                    ha='center', va='bottom', fontsize=9)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=11)
    ax.set_ylabel('Proportion of neurons', fontsize=12)
    
    if show_untuned:
        ax.set_ylim(0, 1.1)
    
    title_map = {
        'social': 'Tuning category proportions across sessions (social trials)',
        'empty': 'Location tuning proportions across sessions (empty trial 1)',
    }
    title = title_map[mode]
    if mode == 'social' and collapse_to_supertypes:
        title = 'Tuning supertype proportions across sessions (social trials)'
    ax.set_title(title, fontsize=13,pad=30)
    
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10, frameon=False)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig, ax

In [ ]:
mean_props_figs_dir_all_sesh = os.path.join(mean_props_dir, "figs_across_all_sessions")
os.makedirs(mean_props_figs_dir_all_sesh, exist_ok=True)

mean_props_figs_dir_gd_sesh = os.path.join(mean_props_dir, "figs_across_good_decoding_sessions")
os.makedirs(mean_props_figs_dir_gd_sesh, exist_ok=True)

In [ ]:
barplot_figsize = (11,6)

Just the tuned cells, with mixed cells colored by identity, for all sessions and only the good decoding sessions

In [ ]:
if save_figs:
    only_tuned_props_identity_color_all_sesh_path = os.path.join(mean_props_figs_dir_all_sesh,'tuning_proportion_bar_graph_only_tuned_cells_mixed_by_identity_all_sessions.svg')
else:
    only_tuned_props_identity_color_all_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props, mode='social',
                                     mixed_color_by='identity',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=False,
                                     save_path=only_tuned_props_identity_color_all_sesh_path, show=True)

In [ ]:
if save_figs:
    only_tuned_props_identity_color_good_decoding_sesh_path = os.path.join(mean_props_figs_dir_gd_sesh,'tuning_proportion_bar_graph_only_tuned_cells_mixed_by_identity_good_decoding_sessions.svg')
else:
    only_tuned_props_identity_color_good_decoding_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props_filtered_social, mode='social',
                                     mixed_color_by='identity',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=False,
                                     save_path=only_tuned_props_identity_color_good_decoding_sesh_path, show=True)

Just the tuned cells, with mixed cells colored by location, for all sessions and only the good decoding sessions

In [ ]:
if save_figs:
    only_tuned_props_location_color_all_sesh_path = os.path.join(mean_props_figs_dir_all_sesh,'tuning_proportion_bar_graph_only_tuned_cells_mixed_by_location_all_sessions.svg')
else:
    only_tuned_props_location_color_all_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=False,
                                     save_path=only_tuned_props_location_color_all_sesh_path, show=True)

In [ ]:
if save_figs:
    only_tuned_props_location_color_good_decoding_sesh_path = os.path.join(mean_props_figs_dir_gd_sesh,'tuning_proportion_bar_graph_only_tuned_cells_mixed_by_location_good_decoding_sessions.svg')
else:
    only_tuned_props_location_color_good_decoding_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props_filtered_social, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=False,
                                     save_path=only_tuned_props_location_color_good_decoding_sesh_path, show=True)

All cells, with mixed cells colored by identity, for all sessions and only the good decoding sessions

In [ ]:
if save_figs:
    all_props_all_sesh_identity_color_path = os.path.join(mean_props_figs_dir_all_sesh,'tuning_proportion_bar_graph_all_cells_mixed_by_identity_all_sessions.svg')
else:
    all_props_all_sesh_identity_color_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props, mode='social',
                                     mixed_color_by='identity',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=True,
                                     save_path=all_props_all_sesh_identity_color_path, show=True)

In [ ]:
if save_figs:
    all_props_good_decoding_sesh_identity_color_path = os.path.join(mean_props_figs_dir_gd_sesh,'tuning_proportion_bar_graph_all_cells_mixed_by_identity_good_decoding_sessions.svg')
else:
    all_props_good_decoding_sesh_identity_color_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props_filtered_social, mode='social',
                                     mixed_color_by='identity',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=True,
                                     save_path=all_props_good_decoding_sesh_identity_color_path, show=True)

All cells, with mixed cells colored by location, for all sessions and only the good decoding sessions

In [ ]:
if save_figs:
    all_props_all_sesh_location_color_path = os.path.join(mean_props_figs_dir_all_sesh,'tuning_proportion_bar_graph_all_cells_mixed_by_location_all_sessions.svg')
else:
    all_props_all_sesh_location_color_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=True,
                                     save_path=all_props_all_sesh_location_color_path, show=True)

In [ ]:
if save_figs:
    all_props_good_decoding_sesh_location_color_path = os.path.join(mean_props_figs_dir_gd_sesh,'tuning_proportion_bar_graph_all_cells_mixed_by_location_good_decoding_sessions.svg')
else:
    all_props_good_decoding_sesh_location_color_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props_filtered_social, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=False,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=True,
                                     save_path=all_props_good_decoding_sesh_location_color_path, show=True)

All cells with collapsed categories, for all sessions and only the good decoding sessions

In [ ]:
if save_figs:
    all_cells_collapsed_props_all_sesh_path = os.path.join(mean_props_figs_dir_all_sesh,'tuning_proportion_bar_graph_all_cells_collapsed_categories_all_sessions.svg')
else:
    all_cells_collapsed_props_all_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=True,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=True,
                                     save_path=all_cells_collapsed_props_all_sesh_path, show=True)

In [ ]:
if save_figs:
    all_cells_collapsed_props_good_decoding_sesh_path = os.path.join(mean_props_figs_dir_gd_sesh,'tuning_proportion_bar_graph_all_cells_collapsed_categories_good_decoding_sessions.svg')
else:
    all_cells_collapsed_props_good_decoding_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props_filtered_social, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=True,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=True,
                                     save_path=all_cells_collapsed_props_good_decoding_sesh_path, show=True)

Just the tuned cells with collapsed categories, for all sessions and only the good decoding sessions

In [ ]:
if save_figs:
    tuned_cells_collapsed_props_all_sesh_path = os.path.join(mean_props_figs_dir_all_sesh,'tuning_proportion_bar_graph_tuned_cells_collapsed_categories_all_sessions.svg')
else:
    tuned_cells_collapsed_props_all_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=True,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=False,
                                     save_path=tuned_cells_collapsed_props_all_sesh_path, show=True)

In [ ]:
if save_figs:
    tuned_cells_collapsed_props_good_decoding_sesh_path = os.path.join(mean_props_figs_dir_gd_sesh,'tuning_proportion_bar_graph_tuned_cells_collapsed_categories_good_decoding_sessions.svg')
else:
    tuned_cells_collapsed_props_good_decoding_sesh_path = None

In [ ]:
fig, ax = plot_tuning_proportions_stacked(summ_tuning_props_filtered_social, mode='social',
                                     mixed_color_by='location',
                                     collapse_to_supertypes=True,
                                     proportion_method='total',
                                     show_errorbars=False,
                                     session_indices=None, figsize=barplot_figsize,
                                     show_n=True, show_untuned=False,
                                     save_path=tuned_cells_collapsed_props_good_decoding_sesh_path, show=True)

## Stats

#### 1) for different pairs of sessions, do tuned proportions of neurons change on average within animals across sessions?
--> Wilcoxon signed-rank test on different session pairs with multiple comparisons correction

In [ ]:
paired_comps_dir = os.path.join(tuning_xsession_dir, "tuning_proportions_across_session_pairs")
os.makedirs(paired_comps_dir, exist_ok=True)

In [ ]:
all_props

In [ ]:
social_categories

In [ ]:
#as defined in session_config.py
comparisons_of_interest = ['same_sex', 'os_presep', 'novel', 'familiar_presep', 'os_familiar']

In [ ]:
#function to perform a paired comparison (via wilcoxon or paired t-test) between the proportion of every tuned cell type across multiple different session pairs
#with benjamini hochberg correction after
def statistical_comparison_over_sessions_cell_proportions(props_df,comparisons_of_interest,categories,test='wilcoxon',alternative='two-sided'): #categories = social_categories
    all_session_pair_comparisons = {}

    for cat in categories:
        cat_df = props_df[props_df['category'] == cat].rename(columns={'proportion': 'value'}) #rename to value to be more appropriate for paired_comparison

        rows = []
        for comp in comparisons_of_interest:
            res_dict = paired_comparison(cat_df, session_indices=SESSION_SUBSETS[comp],
                                          test=test, alternative=alternative)

            if res_dict is None:
                print(f"Skipping category '{cat}' comparison '{comp}': insufficient paired data")
                continue

            row = {k: v for k, v in res_dict.items() if k != 'data'}
            row['comparison_name'] = comp
            row['tuning_category'] = cat
            rows.append(row)

        if len(rows) == 0:
            all_session_pair_comparisons[cat] = pd.DataFrame()
            continue

        res_df = pd.DataFrame(rows)
        rejected, p_corr, _, _ = multipletests(res_df['p_value'], alpha=0.05, method='fdr_bh')
        res_df['p_fdr'] = p_corr
        res_df['sig_after_correction'] = rejected
        res_df = res_df.rename(columns={'p_value': 'p_uncorrected'})

        all_session_pair_comparisons[cat] = res_df

    return all_session_pair_comparisons

In [ ]:
wilcoxon_comps_per_cat_all_sessions = statistical_comparison_over_sessions_cell_proportions(all_props,comparisons_of_interest,categories=social_categories,test='wilcoxon',alternative='two-sided')
wilcoxon_comps_per_cat_all_sessions['right_mixed']

In [ ]:
wilcoxon_comps_per_cat_gd_sessions = statistical_comparison_over_sessions_cell_proportions(all_props_filtered_social,comparisons_of_interest,categories=social_categories,test='wilcoxon',alternative='two-sided')
wilcoxon_comps_per_cat_gd_sessions['novel_cells']

In [ ]:
all_sessions_paired_comps_dir = os.path.join(paired_comps_dir,'all_sessions')
os.makedirs(all_sessions_paired_comps_dir, exist_ok=True)

gd_sessions_paired_comps_dir = os.path.join(paired_comps_dir,'good_decoding_sessions')
os.makedirs(gd_sessions_paired_comps_dir, exist_ok=True)

In [ ]:
if save_csvs:
    for cat in social_categories:
        wilcoxon_comps_per_cat_all_sessions[cat].to_csv(os.path.join(all_sessions_paired_comps_dir,f'{cat}_proportion_across_paired_sessions_wilcoxon_all_sessions.csv'),index=False)
        wilcoxon_comps_per_cat_gd_sessions[cat].to_csv(os.path.join(gd_sessions_paired_comps_dir,f'{cat}_proportion_across_paired_sessions_wilcoxon_good_decoding_sessions.csv'),index=False)

Plot each cell type's proportion across sessions

In [ ]:
def plot_tuning_trajectory(tuning_df, category, session_indices=None,
                            ymin=None, ymax=None, color=None,
                            figsize=(8, 4), title=None,
                            show_individual_animals=True, print_ns=False,
                            save_path=None, show=True):
    """
    Line plot of a single tuning category's proportion across sessions, with
    group mean ± SEM in bold and individual animal trajectories as lighter lines.
    
    Parameters
    ----------
    tuning_df : pd.DataFrame
        Per-animal tuning proportions (output of extract_tuning_proportions()),
        NOT the summarized version. Must contain columns: animal, session_idx,
        short_label, category, proportion.
    category : str
        Which tuning category to plot (e.g. 'partner_cells', 'novel_cells',
        'partner_left_mixed', 'left_empty1_cells').
    session_indices : list[int] or None
        Sessions to include, in order along the x-axis. Defaults to all 
        sessions present in tuning_df for the chosen category.
    ymin, ymax : float or None
        Y-axis limits. If None, matplotlib auto-scales.
    color : str or None
        Color for the group mean line and individual animal lines. If None,
        a category-appropriate default is used.
    figsize : tuple
    title : str or None
        Plot title. If None, auto-generated from category.
    show_individual_animals : bool
        If True, draw a faint line per animal connecting their per-session 
        proportions.
    save_path : str or None
    show : bool
    
    Returns
    -------
    fig, ax
    """
    # Default colors by category
    DEFAULT_COLORS = {
        'partner_cells': PARTNER_COLOR,
        'novel_cells': NOVEL_COLOR,
        'left_cells': LEFT_COLOR,
        'right_cells': RIGHT_COLOR,
        'partner_left_mixed': PARTNER_MIXED_COLOR,
        'partner_right_mixed': PARTNER_MIXED_COLOR,
        'novel_left_mixed': NOVEL_MIXED_COLOR,
        'novel_right_mixed': NOVEL_MIXED_COLOR,
        'partner_mixed': PARTNER_MIXED_COLOR,
        'novel_mixed': NOVEL_MIXED_COLOR,
        'left_mixed': LEFT_MIXED_COLOR,
        'right_mixed': RIGHT_MIXED_COLOR,
        'left_empty1_cells': LEFT_COLOR,
        'right_empty1_cells': RIGHT_COLOR,
    }
    if color is None:
        color = DEFAULT_COLORS.get(category, '#4B4B4B')
    
    # Filter to the requested category
    df = tuning_df[tuning_df['category'] == category].copy()
    if len(df) == 0:
        print(f"No data for category='{category}'")
        return None, None
    
    if session_indices is None:
        session_indices = sorted(df['session_idx'].unique())
    df = df[df['session_idx'].isin(session_indices)]
    
    if len(df) == 0:
        print(f"No data for category='{category}' in the specified sessions")
        return None, None
    
    # Pivot: animals as rows, sessions as columns
    pivot = df.pivot(index='animal', columns='session_idx', values='proportion')
    pivot = pivot.reindex(columns=session_indices)
    
    # Compute group mean and SEM per session (ignoring NaNs)
    means = pivot.mean(axis=0, skipna=True)
    sems = pivot.sem(axis=0, skipna=True)  # pandas SEM uses ddof=1 by default
    n_per_session = pivot.notna().sum(axis=0)
    
    # X positions
    x_pos = np.arange(len(session_indices))
    x_labels = [SESSION_CONFIG['short_labels'][s] for s in session_indices]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Individual animal lines
    if show_individual_animals:
        for animal in pivot.index:
            animal_vals = pivot.loc[animal].values
            # Mask NaNs so the line connects only valid points
            valid = ~np.isnan(animal_vals)
            if valid.sum() > 0:
                ax.plot(x_pos[valid], animal_vals[valid],
                        color=color, alpha=0.4, linewidth=1.5,
                        marker='o', markersize=8, label=animal)
    
    # Group mean ± SEM
    valid_mean = ~means.isna()
    ax.errorbar(x_pos[valid_mean.values], means[valid_mean.values],
                yerr=sems[valid_mean.values],
                fmt='s-', color='black', markerfacecolor='black',
                markersize=12, linewidth=2.5, capsize=5, capthick=2, zorder=10,
                label=f'Group mean ± SEM')
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=11)
    ax.set_xlabel('Timepoint', fontsize=12)
    ax.set_ylabel('Proportion of recorded neurons', fontsize=12)
    
    if ymin is not None or ymax is not None:
        current_ylim = ax.get_ylim()
        new_ymin = ymin if ymin is not None else current_ylim[0]
        new_ymax = ymax if ymax is not None else current_ylim[1]
        ax.set_ylim(new_ymin, new_ymax)
    
    if title is None:
        # Pretty-print the category name
        title = f"{category.replace('_', ' ').title()} proportion across timepoints"
    ax.set_title(title, fontsize=13)
    
    # Annotate n per session below the x-axis tick labels (optional, subtle)
    if print_ns:
        for i, s_idx in enumerate(session_indices):
            n = int(n_per_session.iloc[i])
            ax.text(x_pos[i], ax.get_ylim()[0], f'n={n}',
                    ha='center', va='top', fontsize=8, color='gray',
                    transform=ax.get_xaxis_transform())
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig, ax

In [ ]:
#loop through the different categories for both all sessions and just the sessions with enough pseudotrials for decoding
#also make this function able to pass different session pairs
def plot_tuning_trajectory_loop(props_df,categories,session_indices,ymin=None,ymax=None,figsize=(8,4),save_figs=None,which_sesh='all'):
    '''
    
    which_sesh: 'all' means the df input as props_df has all recording sessions; 'gd' means the df only has the 'good' sessions from decoding (those with sufficient pseudotrials)
    
    '''
    if which_sesh not in ['all','gd']:
        raise ValueError(f"'which_sesh' must be one of 'all' or 'gd', got {which_sesh}")

    if save_figs is None:
        save_figs = globals().get('save_figs', False)
    
    for cat in categories:
        if save_figs:
            if which_sesh == 'all':
                save_path = os.path.join(all_sessions_paired_comps_dir,f'{cat}_tuning_proportion_across_timepoints_all_sessions.svg')
            elif which_sesh == 'gd':
                save_path = os.path.join(gd_sessions_paired_comps_dir,f'{cat}_tuning_proportion_across_timepoints_good_decoding_sessions.svg')
        else:
            save_path = None
            
        fig, ax = plot_tuning_trajectory(props_df, category=cat, session_indices=session_indices,
                            ymin=ymin, ymax=ymax, color=None,
                            figsize=figsize, title=None,
                            show_individual_animals=True, print_ns=False,
                            save_path=save_path, show=True)

In [ ]:
ymin_trajectory_plots = -0.003
ymax_trajectory_plots = 0.17

In [ ]:
plot_tuning_trajectory_loop(all_props,categories=social_categories,session_indices=None,ymin=ymin_trajectory_plots,ymax=ymax_trajectory_plots,
                            save_figs=None,which_sesh='all')

In [ ]:
plot_tuning_trajectory_loop(all_props_filtered_social,categories=social_categories,session_indices=None,ymin=ymin_trajectory_plots,ymax=ymax_trajectory_plots,
                            save_figs=None,which_sesh='gd')

#### 2)  are there more location-tuned neurons than identity-tuned neurons at any/every time point?

In [ ]:
within_sesh_comps_dir = os.path.join(tuning_xsession_dir, "tuning_proportions_compared_within_sessions")
os.makedirs(within_sesh_comps_dir, exist_ok=True)

all_sessions_within_sesh_comps_dir = os.path.join(within_sesh_comps_dir,'all_sessions')
os.makedirs(all_sessions_within_sesh_comps_dir, exist_ok=True)

gd_sessions_within_sesh_comps_dir = os.path.join(within_sesh_comps_dir,'good_decoding_sessions')
os.makedirs(gd_sessions_within_sesh_comps_dir, exist_ok=True)

Generate a dataframe that collapses the tuning types into supertypes (identity-tuned, location-tuned, mixed-selective) and reports the proportion of each supertype across sessions

In [ ]:
supertype_proportions_all_sessions = compute_supertype_proportions(all_props)
supertype_proportions_gd_sessions = compute_supertype_proportions(all_props_filtered_social)
supertype_proportions_all_sessions

In [ ]:
#save out copies
if save_csvs:
    supertype_proportions_all_sessions.to_csv(os.path.join(sc_xsession_dir,'supertype_proportions_all_animals_all_sessions.csv'),index=False)
    supertype_proportions_gd_sessions.to_csv(os.path.join(sc_xsession_dir,'supertype_proportions_all_animals_good_decoding_sessions.csv'),index=False)

In [ ]:
#summarize supertype proportions across all animals
supertype_summary_all_sessions = summarize_tuning_proportions(supertype_proportions_all_sessions)
#supertype_summary_gd_sessions = summarize_tuning_proportions(supertype_proportions_gd_sessions)
supertype_summary_all_sessions

In [ ]:
save_csvs = True

In [ ]:
#save out copies
if save_csvs:
    supertype_summary_all_sessions.to_csv(os.path.join(sc_xsession_dir,'supertype_proportions_summary_all_animals_all_sessions.csv'),index=False)
    supertype_summary_gd_sessions.to_csv(os.path.join(sc_xsession_dir,'supertype_proportions_summary_all_animals_good_decoding_sessions.csv'),index=False)

In [ ]:
#supertype dicts

identity_dict = {'cat_name':'identity_supertype',
                 'cat_label':'identity'}

location_dict = {'cat_name':'location_supertype',
                 'cat_label':'location'}

mixed_dict = {'cat_name':'mixed_supertype',
              'cat_label':'mixed'}

#normal tuning type dicts
#location cells
left_dict = {'cat_name':'left_cells',
             'cat_label':'left'}

right_dict = {'cat_name':'right_cells',
             'cat_label':'right'}

#identity cells
partner_dict = {'cat_name':'partner_cells',
                'cat_label':'partner'}

novel_dict = {'cat_name':'novel_cells',
              'cat_label':'novel'}

#mixed cells
partner_mixed_dict = {'cat_name':'partner_mixed',
                      'cat_label':'partner_mixed'}

novel_mixed_dict = {'cat_name':'novel_mixed',
                    'cat_label':'novel_mixed'}

left_mixed_dict = {'cat_name':'left_mixed',
                    'cat_label':'left_mixed'}

right_mixed_dict = {'cat_name':'right_mixed',
                    'cat_label':'right_mixed'}

In [ ]:
def compare_tuning_props_within_sessions(props_df,type_dict_A,type_dict_B,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all'):

    if which_sesh not in ['all','gd']:
        raise ValueError(f"'which_sesh' must be one of 'all' or 'gd', got {which_sesh}")
    
    if collapse_supertypes:
        df = compute_supertype_proportions(props_df).rename(columns={'proportion':'value'})
    else:
        df = props_df.copy().rename(columns={'proportion':'value'})

    #get the dataframes for just the category in each supertype dictionary
    df_A = df[df['category']==type_dict_A['cat_name']]
    df_B = df[df['category']==type_dict_B['cat_name']]

    res_df = compare_metrics_within_sessions(df_A,df_B,label_a=type_dict_A['cat_label'],label_b=type_dict_B['cat_label'],
                                             session_indices=session_indices,test=test,alternative=alternative,addl_info=None,alpha=0.05)

    if save_csvs is None:
        save_csvs = globals().get('save_csvs', False)

    if save_csvs:
        if which_sesh=='all':
            res_df.to_csv(os.path.join(all_sessions_within_sesh_comps_dir,f"{type_dict_A['cat_label']}_cells_vs_{type_dict_B['cat_label']}_cells_{test}_each_timepoint_all_sessions.csv"),index=False)
        elif which_sesh=='gd':
            res_df.to_csv(os.path.join(gd_sessions_within_sesh_comps_dir,f"{type_dict_A['cat_label']}_cells_vs_{type_dict_B['cat_label']}_cells_{test}_each_timepoint_good_decoding_sessions.csv"),index=False)
    
    return(res_df)

Pure identity-tuned cells vs. pure location-tuned cells

In [ ]:
compare_tuning_props_within_sessions(all_props,identity_dict,location_dict,collapse_supertypes=True,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all')

In [ ]:
compare_tuning_props_within_sessions(all_props_filtered_social,identity_dict,location_dict,collapse_supertypes=True,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='gd')

Pure identity-tuned cells vs. mixed-selective cells

In [ ]:
compare_tuning_props_within_sessions(all_props,identity_dict,mixed_dict,collapse_supertypes=True,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all')

In [ ]:
compare_tuning_props_within_sessions(all_props_filtered_social,identity_dict,mixed_dict,collapse_supertypes=True,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='gd')

Pure location-tuned cells vs. mixed-selective cells

In [ ]:
compare_tuning_props_within_sessions(all_props,location_dict,mixed_dict,collapse_supertypes=True,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all')

In [ ]:
compare_tuning_props_within_sessions(all_props_filtered_social,location_dict,mixed_dict,collapse_supertypes=True,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='gd')

Left-tuned cells vs. right-tuned cells

In [ ]:
compare_tuning_props_within_sessions(all_props,left_dict,right_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all')

In [ ]:
compare_tuning_props_within_sessions(all_props_filtered_social,left_dict,right_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='gd')

Partner-tuned vs. novel-tuned cells

In [ ]:
compare_tuning_props_within_sessions(all_props,partner_dict,novel_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all')

In [ ]:
compare_tuning_props_within_sessions(all_props_filtered_social,partner_dict,novel_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='gd')

Left-mixed vs. right-mixed cells

In [ ]:
compare_tuning_props_within_sessions(all_props,left_mixed_dict,right_mixed_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all')

In [ ]:
compare_tuning_props_within_sessions(all_props_filtered_social,left_mixed_dict,right_mixed_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='gd')

Partner-mixed vs. novel-mixed cells

In [ ]:
compare_tuning_props_within_sessions(all_props,partner_mixed_dict,novel_mixed_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='all')

In [ ]:
compare_tuning_props_within_sessions(all_props_filtered_social,partner_mixed_dict,novel_mixed_dict,collapse_supertypes=False,
                                         session_indices=None,test='wilcoxon',alternative='two-sided',save_csvs=None,which_sesh='gd')

#### 3) Correlate the proportion of each cell type vs. the partner preference score of each animal at the sessions with partner preference tests

Create results path

In [ ]:
props_vs_ppt_path = os.path.join(sc_xsession_dir,'ppt_scores_vs_tuning_proportions')
os.makedirs(props_vs_ppt_path, exist_ok=True)

Import partner preference scores and reshape to tidy format

In [ ]:
PPT_scores_path = r"\\penc8.rc.int.colorado.edu\DonaldsonLab\Sheeran\experiments\hippocampus miniscope imaging\imaging animals ppt results\imaging_animal_ppt_scores.xlsx"

In [ ]:
good_imaging_animals = [k for k, v in ANIMAL_PATHS.items()]
good_imaging_animals

In [ ]:
#loading and formatting
imaging_ppt_scores = pd.read_excel(PPT_scores_path,sheet_name=0)
imaging_ppt_scores['Animal']=imaging_ppt_scores['Animal'].astype(str)
imaging_ppt_scores_good = imaging_ppt_scores[imaging_ppt_scores['Animal'].isin(good_imaging_animals)].copy().reset_index(drop=True)
imaging_ppt_scores_good

In [ ]:
#reshape to tidy format
tidy_ppt_scores = make_ppt_lookup(imaging_ppt_scores_good, score_types=('cup', 'chamber'))
tidy_ppt_scores

In [ ]:
all_props

append ppt scores to each proportion row and return only proportions from sessions with ppts

In [ ]:
props_w_ppt_scores = all_props.merge(tidy_ppt_scores,on=['animal','session_idx'],how='left').dropna(subset=['proportion','ppref_score_cup']).reset_index(drop=True)
props_w_ppt_scores

In [ ]:
props_w_ppt_scores_gd_sesh = all_props_filtered_social.merge(tidy_ppt_scores,on=['animal','session_idx'],how='left').dropna(subset=['proportion','ppref_score_cup']).reset_index(drop=True)
props_w_ppt_scores_gd_sesh

In [ ]:
if save_csvs:
    props_w_ppt_scores.to_csv(os.path.join(props_vs_ppt_path,'all_animals_ppt_sessions_tuning_proportions_vs_ppt_scores_all_sessions.csv'),index=False)
    props_w_ppt_scores_gd_sesh.to_csv(os.path.join(props_vs_ppt_path,'all_animals_ppt_sessions_tuning_proportions_vs_ppt_scores_good_decoding_sessions.csv'),index=False)

loop to perform pearson correlation within each session for ppt score vs. each category's proportion, with benjamini-hochberg correction across sessions

In [ ]:
ppt_correlation_results_all_sesh = {}
ppt_correlation_results_gd_sesh = {}

for cat in social_categories:
    df_all_sesh = props_w_ppt_scores[props_w_ppt_scores['category']==cat]
    res_all_sesh = correlate_across_within_timepoints(df_all_sesh,x_col='ppref_score_cup',y_col='proportion',comparison='within_sessions',session_indices=None,min_n=3,alpha=0.05)
    ppt_correlation_results_all_sesh[cat]=res_all_sesh
    if save_csvs:
        res_all_sesh.to_csv(os.path.join(props_vs_ppt_path,f'{cat}_proportions_vs_ppt_scores_pearsonr_per_timepoint_all_sessions.csv'),index=False)

    df_gd_sesh = props_w_ppt_scores_gd_sesh[props_w_ppt_scores_gd_sesh['category']==cat]
    res_gd_sesh = correlate_across_within_timepoints(df_gd_sesh,x_col='ppref_score_cup',y_col='proportion',comparison='within_sessions',session_indices=None,min_n=3,alpha=0.05)
    ppt_correlation_results_gd_sesh[cat]=res_gd_sesh
    if save_csvs:
        res_gd_sesh.to_csv(os.path.join(props_vs_ppt_path,f'{cat}_proportions_vs_ppt_scores_pearsonr_per_timepoint_good_decoding_sessions.csv'),index=False)

In [ ]:
social_categories

In [ ]:
ppt_correlation_results_all_sesh['right_mixed']

In [ ]:
def plot_proportions_vs_metric(df, behavior_metric='ppref_score_cup',
                               session_indices=None, figsize=(6, 5), 
                               proportion_type_label=None, save_path=None):
    """
    Plot decoding performance vs a metric with coloring by session.

    """
    all_sessions_requested = session_indices is None
    
    if session_indices is None:
        session_indices = sorted(df['session_idx'].unique())
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Color by session
    inds_df_list = []
    for s_idx in session_indices:
        s_data = df[df['session_idx'] == s_idx]
        color = SESSION_CONFIG['colors'][s_idx]
        label = SESSION_CONFIG['short_labels'][s_idx]
        ax.scatter(s_data[behavior_metric], s_data['proportion'],
                  c=color, s=80, alpha=0.7, label=label, edgecolors='white')
        inds_df_list.append(s_data)
    inds_df = pd.concat(inds_df_list).dropna(subset=[behavior_metric, 'proportion'])

    # Correlation
    if len(inds_df) > 2:
        r, p = stats.pearsonr(inds_df[behavior_metric], inds_df['proportion'])
        ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.3f}',
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Regression line
        z = np.polyfit(inds_df[behavior_metric], inds_df['proportion'], 1)
        p_line = np.poly1d(z)
        x_line = np.linspace(inds_df[behavior_metric].min(), inds_df[behavior_metric].max(), 100)
        ax.plot(x_line, p_line(x_line), 'k--', alpha=0.5, linewidth=1)
    
    ax.set_xlabel(behavior_metric.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel('proportion of tuned cells', fontsize=12)
    if proportion_type_label is not None:
        if len(session_indices)==1:
            ax.set_title(f'{proportion_type_label} proportion vs. {behavior_metric}, {SESSION_CONFIG["short_labels"][session_indices[0]]}')
        elif all_sessions_requested:
            ax.set_title(f'{proportion_type_label} proportion vs. {behavior_metric}, all timepoints')
        else:
            ax.set_title(f'{proportion_type_label} proportion vs. {behavior_metric}')
    else:
        if len(session_indices)==1:
            ax.set_title(f'proportion of tuned cells vs. {behavior_metric}, {SESSION_CONFIG["short_labels"][session_indices[0]]}')
        elif all_sessions_requested:
            ax.set_title(f'proportion of tuned cells vs. {behavior_metric}, all timepoints')
        else:
            ax.set_title(f'proportion of tuned cells vs. {behavior_metric}')

    if all_sessions_requested:
        ax.set_title(ax.get_title() + ', all sessions')
    
    ax.legend(loc='best')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    return fig, ax

In [ ]:
ppt_props_figs_path = os.path.join(props_vs_ppt_path,'figures')
os.makedirs(ppt_props_figs_path, exist_ok=True)

In [ ]:
ppt_vs_prop_figsize = (6,5)

In [ ]:
ppt_inds_list = [None,[2],[4],[5]]

In [ ]:
social_categories

In [ ]:
def ppt_vs_proportions_plotting_loop(ppt_vs_props_df,categories,session_index_list=ppt_inds_list,which_sesh='all',
                                  figsize=ppt_vs_prop_figsize,save=None):

    if which_sesh not in ['all','gd']:
        raise ValueError(f"'which_sesh' must be one of 'all' or 'gd', got {which_sesh}")

    if which_sesh == 'all':
        sesh_switch = 'all_sessions'
    elif which_sesh == 'gd':
        sesh_switch = 'good_decoding_sessions'

    if save is None:
        save = globals().get('save_figs', False)
    
    for cat in categories:
        df = ppt_vs_props_df[ppt_vs_props_df['category']==cat]
        
        for idx in session_index_list:
            if save:
                if idx is not None:
                    sesh_type = SESSION_CONFIG['short_labels'][idx[0]]
                    save_path=os.path.join(ppt_props_figs_path,f"ppt_score_vs_{cat}_proportion_{sesh_type}_{sesh_switch}.svg")
                else:
                    save_path=os.path.join(ppt_props_figs_path,f"ppt_score_vs_{cat}_proportion_all_timepoints_{sesh_switch}.svg")
            else:
                save_path=None
            fig, ax = plot_proportions_vs_metric(df, behavior_metric='ppref_score_cup',
                                    session_indices=idx, figsize=figsize, proportion_type_label=cat,save_path=save_path)

In [ ]:
ppt_vs_proportions_plotting_loop(props_w_ppt_scores,categories=social_categories,session_index_list=ppt_inds_list,which_sesh='all',
                                  figsize=ppt_vs_prop_figsize,save=None)

In [ ]:
ppt_vs_proportions_plotting_loop(props_w_ppt_scores_gd_sesh,categories=social_categories,session_index_list=ppt_inds_list,which_sesh='gd',
                                  figsize=ppt_vs_prop_figsize,save=None)